# 🐾 ระบบดึงข้อมูลสายพันธุ์สัตว์เลี้ยงลูกด้วยนม (Mammalia Species Data Retrieval)

สมุดบันทึกนี้ออกแบบมาสำหรับใช้งานบน **Google Colab** เพื่อดึงข้อมูลสายพันธุ์สัตว์เลี้ยงลูกด้วยนม (Mammals) จากฐานข้อมูลระดับโลก **GBIF (Global Biodiversity Information Facility) API** ซึ่งใช้งานได้ฟรีโดยไม่ต้องมี API Key

---

### 1. ติดตั้งและนำเข้าไลบรารีที่จำเป็น (Import Libraries)

In [7]:
import pandas as pd
import numpy as np
import requests
import os
import json
import shutil
import random
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
# from google.colab import files # สำหรับดาวน์โหลดไฟล์บน Colab


# 1. เชื่อมต่อกับ Google Drive (ยกเลิก Comment เมื่อรันบน Colab จริง)
from google.colab import drive
drive.mount('/content/drive')

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow Version: 2.20.0
GPU Available: []


### 2. กำหนดฟังก์ชันดึงข้อมูลสัตว์เลี้ยงลูกด้วยนม
เราจะดึงข้อมูลผ่าน GBIF Species Search API โดยระบุ `classKey=359` (ซึ่งรหัส 359 คือชั้น **Mammalia** หรือสัตว์เลี้ยงลูกด้วยนม)

In [10]:
def fetch_mammal_species(limit=100, offset=0):
    """
    ฟังก์ชันดึงข้อมูลสายพันธุ์สัตว์เลี้ยงลูกด้วยนมจาก GBIF API
    """
    url = "https://api.gbif.org/v1/species/search"
    params = {
        "higherTaxonKey": 359,        # 359 = Mammalia (สัตว์เลี้ยงลูกด้วยนม)
        "rank": "SPECIES",      # ดึงเฉพาะระดับระดับชนิด (Species)
        "status": "ACCEPTED",    # ดึงเฉพาะชื่อที่เป็นที่ยอมรับทางอนุกรมวิธาน
        "limit": limit,
        "offset": offset
    }
    
    try:
        response = requests.get(url, params=params)
        
        # พิมพ์ URL เต็มที่ใช้ส่งออกไป เพื่อนำไปทดสอบบน Browser
        print(f"🔗 Request URL: {response.url}")
        
        response.raise_for_status() # ตรวจสอบ Error
        data = response.json()
        return data.get("results", [])
    except Exception as e:
        print(f"เกิดข้อผิดพลาดในการดึงข้อมูล: {e}")
        return []

### 3. เริ่มดึงข้อมูลและแปลงให้อยู่ในรูปแบบตาราง (Pandas DataFrame)
คุณสามารถปรับเปลี่ยนตัวแปร `total_to_fetch` ด้านล่างเพื่อดึงข้อมูลในจำนวนที่ต้องการได้ (ตัวอย่างนี้ตั้งไว้ที่ 200 รายการ)

In [ ]:
# กำหนดค่าสำหรับการดึงข้อมูลทั้งหมด
limit_per_request = 1000
mammal_list = []
offset = 0

print("กำลังเริ่มต้นดึงข้อมูลทั้งหมดจาก GBIF API...")

while True:
    print(f"ดึงข้อมูลตำแหน่งที่ {offset} ถึง {offset + limit_per_request}...")
    results = fetch_mammal_species(limit=limit_per_request, offset=offset)
    
    if not results:
        break
        
    mammal_list.extend(results)
    
    # หากจำนวนผลลัพธ์ที่ได้กลับมาน้อยกว่า limit แสดงว่าดึงครบแล้ว
    if len(results) < limit_per_request:
        break
        
    offset += limit_per_request
    
    # หน่วงเวลา 0.1 วินาทีเพื่อถนอม API Server
    import time
    time.sleep(0.1)

print(f"✅ ดึงข้อมูลเสร็จสิ้น! ได้รับข้อมูลทั้งหมด {len(mammal_list)} รายการ")

# บันทึกข้อมูลที่ดึงได้ลงไฟล์ JSON ใน Google Drive
import os
import json

output_dir = "/content/drive/MyDrive/Colab Notebooks/SmartZoo/data/species"
os.makedirs(output_dir, exist_ok=True)
json_file_path = os.path.join(output_dir, "mammal_species_data.json")

with open(json_file_path, "w", encoding="utf-8") as f:
    json.dump(mammal_list, f, ensure_ascii=False, indent=2)

print(f"💾 บันทึกข้อมูลทั้งหมดในรูปแบบ JSON เรียบร้อยแล้วที่: {json_file_path}")

### 4. ดึงข้อมูลจำนวนพบบันทึกจริงจาก Occurrence Facets API (Fetch Occurrence Counts)

เนื่องจาก GBIF Species Search API ไม่ได้อัปเดตจำนวนการบันทึกภาพ/พิกัด (ส่งค่า `numOccurrences: 0` ตลอด) เราจึงต้องดึงข้อมูลตัวเลขบันทึกจริงผ่าน **Occurrence Search Facets API** ในขั้นตอนนี้แทน แล้วนำมารวมกับข้อมูลสายพันธุ์ในไฟล์ JSON หลัก

In [16]:
# ดึงข้อมูลจำนวนการบันทึก (Occurrence counts) ผ่าน Facets API เพื่อความแม่นยำและรวดเร็ว
import os
import json
import requests

# โหลดข้อมูลสายพันธุ์ทั้งหมดจากไฟล์เดิม
json_file_path = "/content/drive/MyDrive/Colab Notebooks/SmartZoo/data/species/mammal_species_data.json"

if not os.path.exists(json_file_path):
    data_to_update = mammal_list if 'mammal_list' in locals() else []
else:
    with open(json_file_path, "r", encoding="utf-8") as f:
        data_to_update = json.load(f)

print("กำลังดึงจำนวนบันทึก (Occurrences) จาก Occurrence Facets API...")
try:
    # classKey=359 คือกลุ่ม Mammalia (สัตว์เลี้ยงลูกด้วยนม)
    facet_url = "https://api.gbif.org/v1/occurrence/search?classKey=359&facet=speciesKey&facetLimit=50000&limit=0"
    headers = {"User-Agent": "SmartZooResearchBot/1.0 (your-email@example.com)"}
    response = requests.get(facet_url, headers=headers, timeout=15)
    
    if response.status_code == 200:
        facet_data = response.json()
        # แปลงข้อมูลเป็น Dict {speciesKey: count}
        counts_map = {
            item["name"]: item["count"] 
            for item in facet_data.get("facets", [])[0].get("counts", [])
        }
        
        # นำจำนวนการบันทึกมา Map ใส่ในข้อมูลสายพันธุ์
        for sp in data_to_update:
            sp_key = str(sp.get("key"))
            sp["numberOfOccurrences"] = counts_map.get(sp_key, 0)
        print("✅ ดึงและจับคู่จำนวน Occurrences สำเร็จ!")
    else:
        print(f"⚠️ เกิดข้อผิดพลาดจาก API (Status code: {response.status_code})")
        for sp in data_to_update:
            sp["numberOfOccurrences"] = 0
except Exception as e:
    print(f"⚠️ เกิดข้อผิดพลาดในการดึง Facets: {e}")
    for sp in data_to_update:
        sp["numberOfOccurrences"] = 0

# บันทึกข้อมูลที่อัปเดตแล้วทับไฟล์ JSON เดิม
with open(json_file_path, "w", encoding="utf-8") as f:
    json.dump(data_to_update, f, ensure_ascii=False, indent=2)

print(f"💾 อัปเดตและบันทึกข้อมูลเรียบร้อยแล้วที่: {json_file_path}")

กำลังดึงจำนวนบันทึก (Occurrences) จาก Occurrence Facets API...
✅ ดึงและจับคู่จำนวน Occurrences สำเร็จ!
💾 อัปเดตและบันทึกข้อมูลเรียบร้อยแล้วที่: /content/drive/MyDrive/Colab Notebooks/SmartZoo/data/species/mammal_species_data.json


### 5. คัดกรองสายพันธุ์ที่มีการบันทึกมากกว่า 100 ครั้ง (Filter Species with > 100 Occurrences)

เราจะคัดเอาเฉพาะสายพันธุ์สัตว์เลี้ยงลูกด้วยนมที่มีจำนวนการพบบันทึก (`numberOfOccurrences`) มากกว่า 100 ครั้งขึ้นไป และบันทึกข้อมูลผลลัพธ์ลงเป็นไฟล์ JSON ตัวใหม่สำหรับใช้ไปสอบถามหาภาพและดาวน์โหลด

In [17]:
# โหลดข้อมูลสายพันธุ์ทั้งหมดจากไฟล์เดิม
import os
import json

json_file_path = "/content/drive/MyDrive/Colab Notebooks/SmartZoo/data/species/mammal_species_data.json"

if not os.path.exists(json_file_path):
    data_to_filter = data_to_update if 'data_to_update' in locals() else []
else:
    with open(json_file_path, "r", encoding="utf-8") as f:
        data_to_filter = json.load(f)

# ทำการกรองเฉพาะสายพันธุ์ที่มี numberOfOccurrences > 100
filtered_mammals = [
    sp for sp in data_to_filter
    if sp.get("numberOfOccurrences") is not None
    and sp.get("numberOfOccurrences") > 100
]

print(f"📊 จำนวนสายพันธุ์ทั้งหมด: {len(data_to_filter)} รายการ")
print(f"🎯 จำนวนสายพันธุ์หลังการคัดกรอง (> 100 occurrences): {len(filtered_mammals)} รายการ")

# บันทึกลงเป็นไฟล์ JSON ตัวใหม่
output_dir = "/content/drive/MyDrive/Colab Notebooks/SmartZoo/data/species"
os.makedirs(output_dir, exist_ok=True)
filtered_json_path = os.path.join(output_dir, "mammal_species_gt100.json")

with open(filtered_json_path, "w", encoding="utf-8") as f:
    json.dump(filtered_mammals, f, ensure_ascii=False, indent=2)

print(f"💾 บันทึกไฟล์ที่คัดกรองแล้วสำเร็จที่: {filtered_json_path}")

# แสดงตัวอย่าง 5 สายพันธุ์แรกหลังคัดกรอง
if filtered_mammals:
    print("\n🔍 ตัวอย่าง 5 สายพันธุ์แรกที่ผ่านการคัดกรอง:")
    for i, sp in enumerate(filtered_mammals[:5], 1):
        print(f"{i}. {sp.get('scientificName')} (Occurrences: {sp.get('numberOfOccurrences')})")

📊 จำนวนสายพันธุ์ทั้งหมด: 21100 รายการ
🎯 จำนวนสายพันธุ์หลังการคัดกรอง (> 100 occurrences): 4534 รายการ
💾 บันทึกไฟล์ที่คัดกรองแล้วสำเร็จที่: /content/drive/MyDrive/Colab Notebooks/SmartZoo/data/species/mammal_species_gt100.json

🔍 ตัวอย่าง 5 สายพันธุ์แรกที่ผ่านการคัดกรอง:
1. Galeopterus variegatus (Audebert, 1799) (Occurrences: 2692)
2. Oryzorictes tetradactylus Milne-Edwards & A.Grandidier, 1882 (Occurrences: 371)
3. Oryzorictes hova A.Grandidier, 1870 (Occurrences: 154)
4. Microgale thomasi Major, 1896 (Occurrences: 234)
5. Microgale fotsifotsy Jenkins, Raxworthy & Nussbaum, 1997 (Occurrences: 168)
